In [1]:
import boto3
import pandas as pd
from pathlib import PurePosixPath
import re

client = boto3.client("s3")
bucket = "dcceew-eds-data"

# Optional: set a prefix if you know it, otherwise use ""
prefix = ""

paginator = client.get_paginator("list_objects_v2")

rows = []
for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
    for obj in page.get("Contents", []):
        key = obj["Key"]
        name = PurePosixPath(key).name
        suffix = PurePosixPath(key).suffix.lower()
        rows.append({
            "key": key,
            "file": name,
            "suffix": suffix,
            "size_MB": round(obj["Size"] / 1e6, 3),
            "last_modified": obj["LastModified"],
        })

df = pd.DataFrame(rows)

if df.empty:
    print("No objects found.")
else:
    display(df.head(50))
    print(f"Total objects: {len(df)}")

,key,file,suffix,size_MB,last_modified
0,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/manife...,p089r078_manifest.parquet,.parquet,0.006,2026-03-24 22:30:24+00:00
1,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/manife...,p089r079_manifest.parquet,.parquet,0.006,2026-03-04 23:21:05+00:00
2,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/manife...,p089r080_manifest.parquet,.parquet,0.006,2026-03-24 23:21:57+00:00
3,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/...,sl8olre_p089r078_20150212_ga1-clr_e32756.tif,.tif,25.338,2026-03-24 22:36:14+00:00
4,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/...,sl8olre_p089r078_20150212_ga2_e32756.tif,.tif,1.058,2026-03-24 22:36:15+00:00
5,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/...,sl8olre_p089r078_20150228_ga1-clr_e32756.tif,.tif,28.767,2026-03-24 22:36:33+00:00
6,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/...,sl8olre_p089r078_20150228_ga2_e32756.tif,.tif,1.025,2026-03-24 22:36:34+00:00
7,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/...,sl8olre_p089r078_20150316_ga1-clr_e32756.tif,.tif,17.184,2026-03-24 22:36:52+00:00
8,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/...,sl8olre_p089r078_20150316_ga2_e32756.tif,.tif,0.677,2026-03-24 22:36:53+00:00
9,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/...,sl8olre_p089r078_20150503_ga1-clr_e32756.tif,.tif,40.711,2026-03-15 22:28:00+00:00


Total objects: 12359


In [2]:
eds_regex = (
    r"(_dlj_e|dlj-dlj|ga0_)"
)

eds_df = df[df["file"].str.contains(eds_regex, case=False, regex=True)].copy()

display(eds_df.head(100))
print(f"EDS-like objects: {len(eds_df)}")

/tmp/ipykernel_67522/3291618547.py:5: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  eds_df = df[df["file"].str.contains(eds_regex, case=False, regex=True)].copy()


,key,file,suffix,size_MB,last_modified
342,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/...,sl8olre_p089r078_20250701_ga0_e32756.tif,.tif,366.299,2026-04-20 03:38:01+00:00
377,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/...,sl8olre_p089r078_20251208_ga0_e32756.tif,.tif,458.238,2026-03-15 23:17:15+00:00
382,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/...,sl9olre_p089r078_20260117_ga0_e32756.tif,.tif,459.914,2026-04-20 03:40:39+00:00
395,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/...,sl8olre_p089r078_d2025070120251208_dlj-dlj-cle...,.tif,0.882,2026-03-15 23:19:12+00:00
396,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/...,sl8olre_p089r078_d2025070120251208_dlj-dlj-str...,.tif,0.953,2026-03-15 23:19:12+00:00
...,...,...,...,...,...
1576,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/...,sl8olre_p089r081_d2025082620260109_dlj-dlj-cle...,.shp,3.796,2026-03-16 01:29:09+00:00
1577,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/...,sl8olre_p089r081_d2025082620260109_dlj-dlj-cle...,.shx,0.015,2026-03-16 01:29:09+00:00
1578,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/...,sl8olre_p089r081_d2025082620260109_dlj-dlj-str...,.cpg,0.000,2026-03-16 01:29:07+00:00
1579,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/...,sl8olre_p089r081_d2025082620260109_dlj-dlj-str...,.dbf,0.103,2026-03-16 01:29:08+00:00


EDS-like objects: 777


In [3]:
import re

def extract_tile(fname):
    m = re.search(r"p\d{3}r\d{3}", fname.lower())
    return m.group(0) if m else None

eds_df["tile"] = eds_df["file"].apply(extract_tile)

eds_df.head()

,key,file,suffix,size_MB,last_modified,tile
342,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/...,sl8olre_p089r078_20250701_ga0_e32756.tif,.tif,366.299,2026-04-20 03:38:01+00:00,p089r078
377,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/...,sl8olre_p089r078_20251208_ga0_e32756.tif,.tif,458.238,2026-03-15 23:17:15+00:00,p089r078
382,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/...,sl9olre_p089r078_20260117_ga0_e32756.tif,.tif,459.914,2026-04-20 03:40:39+00:00,p089r078
395,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/...,sl8olre_p089r078_d2025070120251208_dlj-dlj-cle...,.tif,0.882,2026-03-15 23:19:12+00:00,p089r078
396,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/...,sl8olre_p089r078_d2025070120251208_dlj-dlj-str...,.tif,0.953,2026-03-15 23:19:12+00:00,p089r078


In [4]:
tile_counts = (
    eds_df.groupby("tile")
    .size()
    .reset_index(name="file_count")
    .sort_values("tile")
)

display(tile_counts)

,tile,file_count
0,p089r078,29
1,p089r079,29
2,p089r080,29
3,p089r081,15
4,p089r082,15
5,p089r083,15
6,p089r084,15
7,p090r077,15
8,p090r078,15
9,p090r079,15


# Print out EDS results

In [5]:
for tile, group in eds_df.sort_values(["tile","file"]).groupby("tile"):
    
    print(f"\n--- {tile} ---")
    
    for f in group["file"]:
        print(f)


--- p089r078 ---
sl8olre_p089r078_20250701_ga0_e32756.tif
sl8olre_p089r078_20251208_ga0_e32756.tif
sl8olre_p089r078_d2025070120251208_dlj-dlj-clear-ge80_e32756.cpg
sl8olre_p089r078_d2025070120251208_dlj-dlj-clear-ge80_e32756.dbf
sl8olre_p089r078_d2025070120251208_dlj-dlj-clear-ge80_e32756.prj
sl8olre_p089r078_d2025070120251208_dlj-dlj-clear-ge80_e32756.shp
sl8olre_p089r078_d2025070120251208_dlj-dlj-clear-ge80_e32756.shx
sl8olre_p089r078_d2025070120251208_dlj-dlj-clear-ge80_e32756.tif
sl8olre_p089r078_d2025070120251208_dlj-dlj-strong-ge60_e32756.cpg
sl8olre_p089r078_d2025070120251208_dlj-dlj-strong-ge60_e32756.dbf
sl8olre_p089r078_d2025070120251208_dlj-dlj-strong-ge60_e32756.prj
sl8olre_p089r078_d2025070120251208_dlj-dlj-strong-ge60_e32756.shp
sl8olre_p089r078_d2025070120251208_dlj-dlj-strong-ge60_e32756.shx
sl8olre_p089r078_d2025070120251208_dlj-dlj-strong-ge60_e32756.tif
sl8olre_p089r078_d2025070120251208_dlj_e32756.tif
sl9olre_p089r078_20260117_ga0_e32756.tif
sl9olre_p089r078_d20250

# Export and zip

In [6]:
from pathlib import Path

export_dir = Path("/home/jovyan/work-easi-eds/output/eds_outputs_from_s3tiles")
export_dir.mkdir(parents=True, exist_ok=True)

export_dir

PosixPath('/home/jovyan/work-easi-eds/output/eds_outputs_from_s3tiles')

In [7]:
row = eds_df.iloc[0]

print("bucket:", bucket)
print("key:", repr(row["key"]))

bucket: dcceew-eds-data
key: 'AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p089r078/2025/20250701/sl8olre_p089r078_20250701_ga0_e32756.tif'


In [8]:
client.head_object(Bucket=bucket, Key=row["key"])

{'ResponseMetadata': {'RequestId': '1GDTZAW9Z9RW70Z0',
  'HostId': '3qsMkSyuTGNR71skNp9/TcD0bKd4SrxOeT+TcnTW2yBilaXw7o3PpgABAuGRlYYdedM0IGFmg5czminpuhPa2iYAVOPD0cUX',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amz-id-2': '3qsMkSyuTGNR71skNp9/TcD0bKd4SrxOeT+TcnTW2yBilaXw7o3PpgABAuGRlYYdedM0IGFmg5czminpuhPa2iYAVOPD0cUX',
   'x-amz-request-id': '1GDTZAW9Z9RW70Z0',
   'date': 'Mon, 20 Apr 2026 06:47:16 GMT',
   'last-modified': 'Mon, 20 Apr 2026 03:38:01 GMT',
   'etag': '"4d5164ecce6ab4dff65f28fb4f4d16e1-44"',
   'x-amz-server-side-encryption': 'AES256',
   'x-amz-version-id': 'null',
   'accept-ranges': 'bytes',
   'content-type': 'binary/octet-stream',
   'content-length': '366298854',
   'server': 'AmazonS3'},
  'RetryAttempts': 0},
 'AcceptRanges': 'bytes',
 'LastModified': datetime.datetime(2026, 4, 20, 3, 38, 1, tzinfo=tzutc()),
 'ContentLength': 366298854,
 'ETag': '"4d5164ecce6ab4dff65f28fb4f4d16e1-44"',
 'VersionId': 'null',
 'ContentType': 'binary/octet-stream',
 'ServerSideE

In [9]:
import re
import shutil
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED

# Settings
delete_unzipped_files = False   # True = delete files after successful zipping

print(f"Output directory: {export_dir.resolve()}")


def extract_date_group(filename: str) -> str:
    """
    Extract a grouping token from filename.

    Examples:
    - sl8olre_p089r078_20250701_ga0_e32756.tif
      -> 20250701

    - sl8olre_p089r078_d2025070120251208_dlj_e32756.tif
      -> d2025070120251208
    """
    # Match date ranges first, e.g. d2025070120251208
    m = re.search(r'(d\d{16})', filename)
    if m:
        return m.group(1)

    # Then match single dates, e.g. 20250701
    m = re.search(r'(?<!\d)(\d{8})(?!\d)', filename)
    if m:
        return m.group(1)

    return "unknown"


for tile, group in eds_df.groupby("tile"):

    tile_dir = export_dir / tile
    tile_dir.mkdir(parents=True, exist_ok=True)

    print(f"\nDownloading {tile}")
    print(f"Tile folder: {tile_dir.resolve()}")

    downloaded_files = []

    for _, row in group.iterrows():
        key = row["key"]
        local_path = tile_dir / row["file"]

        client.download_file(bucket, key, str(local_path))
        downloaded_files.append(local_path)

        print("  downloaded:", local_path.name)

    # Group downloaded files by extracted date token
    files_by_group = {}

    for file_path in downloaded_files:
        group_key = extract_date_group(file_path.name)
        files_by_group.setdefault(group_key, []).append(file_path)

    print(f"\nCreating split ZIPs for {tile}")

    for group_key, file_list in sorted(files_by_group.items()):
        zip_path = export_dir / f"{tile}_{group_key}.zip"
        print(f"  creating: {zip_path.resolve()}")

        with ZipFile(zip_path, "w", compression=ZIP_DEFLATED) as zipf:
            for file_path in file_list:
                # Store file inside zip with just its filename
                zipf.write(file_path, arcname=file_path.name)

        print(f"  zipped: {zip_path.name} ({len(file_list)} files)")

    # Optionally delete originals after all ZIPs are created
    if delete_unzipped_files:
        shutil.rmtree(tile_dir)
        print(f"  deleted folder: {tile_dir}")

Output directory: /home/jovyan/work-easi-eds/output/eds_outputs_from_s3tiles

Tile folder: /home/jovyan/work-easi-eds/output/eds_outputs_from_s3tiles/p089r078
  downloaded: sl8olre_p089r078_20250701_ga0_e32756.tif
  downloaded: sl8olre_p089r078_20251208_ga0_e32756.tif
  downloaded: sl9olre_p089r078_20260117_ga0_e32756.tif
  downloaded: sl8olre_p089r078_d2025070120251208_dlj-dlj-clear-ge80_e32756.tif
  downloaded: sl8olre_p089r078_d2025070120251208_dlj-dlj-strong-ge60_e32756.tif
  downloaded: sl9olre_p089r078_d2025070120260117_dlj-dlj-clear-ge80_e32756.tif
  downloaded: sl9olre_p089r078_d2025070120260117_dlj-dlj-strong-ge60_e32756.tif
  downloaded: sl8olre_p089r078_d2025070120251208_dlj_e32756.tif
  downloaded: sl9olre_p089r078_d2025070120260117_dlj_e32756.tif
  downloaded: sl8olre_p089r078_d2025070120251208_dlj-dlj-clear-ge80_e32756.cpg
  downloaded: sl8olre_p089r078_d2025070120251208_dlj-dlj-clear-ge80_e32756.dbf
  downloaded: sl8olre_p089r078_d2025070120251208_dlj-dlj-clear-ge80_e327

In [ ]:
print("\nOutput directory:")
print(export_dir.resolve())

In [ ]:
import re
import shutil
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED

# SETTINGS
tile_to_process = "p089r078"
delete_unzipped_files = False   # True = delete files after zipping

print("="*60)
print(f"Output directory: {export_dir.resolve()}")
print(f"Processing tile: {tile_to_process}")
print("="*60)


def extract_date_group(filename: str) -> str:
    """
    Extract grouping token from filename.

    Examples:
    20250701
    d2025070120251208
    """
    m = re.search(r'(d\d{16})', filename)
    if m:
        return m.group(1)

    m = re.search(r'(?<!\d)(\d{8})(?!\d)', filename)
    if m:
        return m.group(1)

    return "unknown"


# filter dataframe
group = eds_df[eds_df["tile"] == tile_to_process]

tile_dir = export_dir / tile_to_process
tile_dir.mkdir(parents=True, exist_ok=True)

print(f"\nDownloading files to: {tile_dir.resolve()}")

downloaded_files = []

for _, row in group.iterrows():

    key = row["key"]
    local_path = tile_dir / row["file"]

    # skip if already downloaded
    if not local_path.exists():

        client.download_file(
            bucket,
            key,
            str(local_path)
        )

        print("  downloaded:", local_path.name)

    else:
        print("  exists:", local_path.name)

    downloaded_files.append(local_path)


# group files by date/date-range
files_by_group = {}

for file_path in downloaded_files:

    group_key = extract_date_group(file_path.name)

    files_by_group.setdefault(group_key, []).append(file_path)


print("\nCreating ZIP files:")

for group_key, file_list in sorted(files_by_group.items()):

    zip_path = export_dir / f"{tile_to_process}_{group_key}.zip"

    print(f"  creating {zip_path.name}")

    with ZipFile(zip_path, "w", compression=ZIP_DEFLATED) as zipf:

        for file_path in file_list:

            zipf.write(
                file_path,
                arcname=file_path.name
            )

    print(f"  zipped {len(file_list)} files")


# optionally delete original files
if delete_unzipped_files:

    for f in downloaded_files:

        if f.exists():
            f.unlink()

    print(f"\nDeleted unzipped files from {tile_dir}")